# NDgpu — latest coupled-transient GPU validation

This notebook validates and measures the current GPU coupling path after the transient performance work. It is intentionally a test notebook, not just a physics demonstration.

It checks:

1. the latest API is actually installed;
2. a trailing partial thermal window is advanced;
3. power normalization and temperature telemetry stay device-resident between thermal exchanges;
4. thermal diagnostics cadence and coupled counters work;
5. CPU and GPU follow the same coupled transient;
6. conservative and GPU-oriented solver controls give consistent physics;
7. CUDA-event phase timings separate coupled-equilibrium startup, the transient initial eigen solve, actual neutron marching, operator rebuilds, power edits, thermal solves, and transfers;
8. a deliberately non-trivial 11-group perturbation performs real Krylov work; and
9. an opt-in one-minute 3-D HP-MR drum manoeuvre provides a realistic target for quasi-static acceleration.

> In Colab select **Runtime → Change runtime type → T4 GPU** before running. First run `python tools/build_src_zip.py` in the repository, then upload `dist/ndgpu-src.zip`; the API gate below fails clearly if an older archive is uploaded.


In [ ]:
# Colab install. Local Jupyter runs use the already-importable checkout.
try:
    from google.colab import files
    uploaded = files.upload()
    archive = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q --force-reinstall --no-deps {archive}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().system("nvidia-smi")
except ImportError:
    pass


In [ ]:
import inspect
import time
import numpy as np
import ndgpu

from ndgpu.benchmarks.hpmr import HPMR_KINETICS, build_hpmr2d, build_hpmr3d
from ndgpu.benchmarks.hpmr_thermal import (
    RATED_POWER_W, build_hpmr_coupling, hpmr_angle_for_dollars,
    hpmr_drum_ramp, hpmr_endfb8_builtin, sink_coefficient)
from ndgpu.benchmarks.hpmr_transient_bench import (
    SIGMA_A_SCALE, scale_absorption)
from ndgpu.coupling import coupled_transient

try:
    import cupy as cp
    HAVE_GPU = cp.cuda.runtime.getDeviceCount() > 0
except Exception as exc:
    HAVE_GPU = False
    print("CuPy/CUDA check failed:", exc)

assert HAVE_GPU, "No CUDA GPU is visible. Enable a GPU runtime and rerun from the top."

required = {
    "precond_degree", "check_every", "thermal_rtol",
    "thermal_check_every", "thermal_diagnostics_every", "profile",
}
parameters = set(inspect.signature(coupled_transient).parameters)
missing = required - parameters
assert not missing, (
    f"Installed archive predates the coupled GPU changes; missing {sorted(missing)}. "
    "Rebuild and upload dist/ndgpu-src.zip.")

props = cp.cuda.runtime.getDeviceProperties(cp.cuda.runtime.getDevice())
free_b, total_b = cp.cuda.runtime.memGetInfo()
print("ndgpu:", inspect.getfile(ndgpu))
print("GPU  :", props["name"].decode())
print(f"VRAM : {free_b/2**30:.2f} GiB free / {total_b/2**30:.2f} GiB total")
print("coupled_transient", inspect.signature(coupled_transient))


In [ ]:
def build_case(refine=3, groups="2", nz=0, device="gpu",
               drum=(150.0, 154.0), ramp=0.20, t_start=0.0,
               n_angles=5):
    """Build one fresh coupled problem and its drum-ramp callback."""
    mats = hpmr_endfb8_builtin(three_d=nz > 0) if groups == "11" else None
    build = build_hpmr3d if nz else build_hpmr2d
    kwargs = dict(refine=refine, drum_angle_deg=drum[0],
                  absorber="polar", materials=mats)
    if nz:
        kwargs["nz"] = nz
    problem = build(**kwargs)
    ctx = build_hpmr_coupling(problem, device=device)
    problem_at = hpmr_drum_ramp(
        problem, angle_from=drum[0], angle_to=drum[1],
        t_start=t_start, t_ramp=ramp, n_angles=n_angles, refine=refine, nz=nz,
        materials=mats)
    return problem, ctx, problem_at


def run_case(*, refine=3, groups="2", nz=0, device="gpu",
             t_end=0.30, dt=0.05, dt_thermal=0.25,
             drum=(150.0, 154.0), ramp=0.20, t_start=0.0,
             n_angles=5, **controls):
    problem, ctx, problem_at = build_case(
        refine, groups, nz, device, drum=drum, ramp=ramp,
        t_start=t_start, n_angles=n_angles)
    t0 = time.perf_counter()
    result = coupled_transient(
        ctx, t_end=t_end, dt=dt, dt_thermal=dt_thermal,
        problem_at=problem_at, profile=True, **controls)
    wall = time.perf_counter() - t0
    cells = int(np.count_nonzero(problem.active))
    return result, wall, cells


def print_profile(result, wall=None):
    total = result.phase_seconds.get("transient_total", result.seconds)
    initial = result.phase_seconds.get("initial_eigen_solve", 0.0)
    march = max(total - initial, 0.0)
    print(result)
    if wall is not None:
        print(f"end-to-end wall                         : {wall:.3f} s")
        print(f"startup outside transient profile       : {max(wall-total, 0):.3f} s")
    print(f"  coupled equilibrium                   : {result.steady.seconds:.3f} s  ({result.steady.iterations} iterations)")
    print(f"transient initial critical solve         : {initial:.3f} s")
    print(f"profiled time-march envelope              : {march:.3f} s")
    print(f"profiled transient total                 : {total:.3f} s")
    print(f"max |P/P0 - 1|                          : {np.max(np.abs(result.power-1)):.6e}")
    print("\nexclusive phase                     seconds   % transient")
    for name, seconds in sorted(result.phase_seconds.items()):
        if name == "transient_total":
            continue
        print(f"{name:34s} {seconds:9.4f} {100*seconds/max(total, 1e-30):11.2f}")
    print("\ncounters")
    for name, value in sorted(result.counters.items()):
        print(f"{name:34s} {value:,}")


## 1. GPU regression smoke test

Six neutron steps make one full 0.25 s thermal window plus a trailing 0.05 s window. This exercises the partial-window fix, degree-1 neutron preconditioning, spaced Krylov checks, thermal controls, exact diagnostics on request, phase events, and the reduced telemetry-transfer cadence. The first run also compiles the CUDA kernels.


In [ ]:
smoke, smoke_wall, smoke_cells = run_case(
    refine=3, groups="2", device="gpu",
    t_end=0.30, dt=0.05, dt_thermal=0.25,
    thermal_diagnostics_every=1)
print_profile(smoke, smoke_wall)

assert smoke.steps == 6
assert smoke.counters["neutronics_steps"] == 6
assert smoke.counters["thermal_steps"] == 2
assert smoke.counters["thermal_diagnostics"] == 2
assert smoke.counters["telemetry_transfers"] == 2
assert smoke.counters["result_transfers"] == 1
assert smoke.counters["initial_eigen_outer_iterations"] > 0
assert smoke.counters["initial_eigen_inner_iterations"] > 0
assert smoke.counters["neutron_inner_iterations"] > 0
assert np.all(np.isfinite(smoke.power))
assert np.all(np.isfinite(smoke.temperature))
required_phases = {
    "initial_eigen_solve", "neutronics_solve",
    "operator_rebuild", "power_edit",
    "thermal_solve", "feedback_update", "telemetry_transfer",
    "result_transfer", "transient_total",
}
assert required_phases <= smoke.phase_seconds.keys()
print("\nPASS: latest coupled GPU capabilities are active.")


## 2. CPU/GPU correctness gate

The same short two-group calculation is run on both backends with identical controls. This is deliberately small: it checks arithmetic and coupling semantics, not speed. Different reduction orders can change the last bits.


In [ ]:
common = dict(refine=2, groups="2", t_end=0.10, dt=0.05,
              dt_thermal=0.10, drum=(150.0, 152.0), ramp=0.10)
cpu, cpu_wall, _ = run_case(device="cpu", **common)
gpu, gpu_wall, _ = run_case(device="gpu", **common)

dP = float(np.max(np.abs(cpu.power - gpu.power)))
dT = float(np.max(np.abs(cpu.mean_temperature - gpu.mean_temperature)))
print(f"CPU: {cpu_wall:.3f} s  P(end)={cpu.power[-1]:.10f}  Tmean={cpu.mean_temperature[-1]:.7f} K")
print(f"GPU: {gpu_wall:.3f} s  P(end)={gpu.power[-1]:.10f}  Tmean={gpu.mean_temperature[-1]:.7f} K")
print(f"max |dP/P0|={dP:.3e}; max |dTmean|={dT:.3e} K")
assert dP < 2e-8
assert dT < 2e-6
print("PASS: CPU and GPU coupled histories agree.")


## 3. GPU controls A/B

This compares the old conservative behavior with the new production-oriented controls on the same GPU case. The comparison is intentionally reported rather than asserted: small two-group problems are launch-bound, have very short Krylov solves, and may not benefit from checking every fourth iteration. The important assertions are that both runs execute and agree within their requested solver tolerances.

The conservative leg computes an exact thermal energy balance at every thermal step. The optimized leg avoids those reductions and uses the coupled defaults.


In [ ]:
ab_case = dict(refine=3, groups="2", device="gpu",
               t_end=0.50, dt=0.05, dt_thermal=0.25)
optimized, opt_wall, _ = run_case(**ab_case)
conservative, con_wall, _ = run_case(
    **ab_case, precond_degree=0, check_every=1,
    thermal_rtol=1e-12, thermal_check_every=1,
    thermal_precond_degree=0, thermal_diagnostics_every=1)

opt_t = optimized.phase_seconds["transient_total"]
con_t = conservative.phase_seconds["transient_total"]
dP = float(np.max(np.abs(optimized.power - conservative.power)))
dT = float(np.max(np.abs(optimized.mean_temperature - conservative.mean_temperature)))
print(f"optimized    transient={opt_t:.4f} s  wall={opt_wall:.4f} s  inner={optimized.counters['neutron_inner_iterations']:,}")
print(f"conservative transient={con_t:.4f} s  wall={con_wall:.4f} s  inner={conservative.counters['neutron_inner_iterations']:,}")
print(f"conservative/optimized transient ratio: {con_t/opt_t:.3f}x")
print(f"max history differences: dP={dP:.3e}, dT={dT:.3e} K")
assert dP < 2e-4
assert dT < 2e-4
print("PASS: optimized controls preserve the coupled solution.")
print("\nOptimized profile:")
print_profile(optimized)


## 4. Real 11-group coupled transient

This is the representative multigroup path: real ENDF/B-VIII-derived HP-MR constants, upscatter, six spectral subsweeps, whole-core rebalance, device-resident fission-energy power edit, thermal feedback, and the new GPU defaults. A roughly half-dollar uniform absorption step is used deliberately: unlike the old 150→151° partial drum ramp, it is mesh-insensitive and forces the time marcher to perform real fixed-point and Krylov work. Drum motion is exercised by the full 3-D case below.

`initial_eigen_solve`, time-march `neutronics_solve`, and dynamic `operator_rebuild` are exclusive phases. `transient_total` includes all three plus callbacks. Phase values are CUDA-event elapsed times and do not insert synchronization at every boundary.


In [ ]:
real11_problem, real11_ctx, _ = build_case(
    refine=3, groups="11", device="gpu", drum=(150.0, 150.0))
real11_perturbed = scale_absorption(real11_problem.materials, SIGMA_A_SCALE)
real11_problem_at = lambda t: (
    real11_problem.materials if t <= 0 else real11_perturbed,
    real11_problem.material_map, real11_problem.mix_material,
    real11_problem.mix_weight)
t0 = time.perf_counter()
real11 = coupled_transient(
    real11_ctx, t_end=0.04, dt=0.02, dt_thermal=0.04,
    problem_at=real11_problem_at, profile=True)
real11_wall = time.perf_counter() - t0
real11_cells = int(np.count_nonzero(real11_problem.active))
print(f"active cells: {real11_cells:,}; unknowns: {11*real11_cells:,}")
print_profile(real11, real11_wall)
assert real11.steps == 2
assert real11.counters["thermal_steps"] == 1
assert real11.counters["telemetry_transfers"] == 1
assert real11.counters["neutron_inner_iterations"] > 0
assert max(real11.counters["neutron_fixed_point_sweeps"], 0) > real11.steps
assert np.max(np.abs(real11.power - 1.0)) > 1e-3
assert np.all(np.isfinite(real11.power))
print("\nPASS: 11-group coupled GPU transient completed.")


## 5. Optional one-minute 3-D coupled HP-MR manoeuvre

This is the challenging production-style case, not a smoke test. It uses the real 11-group 3-D core (`refine=4`, `nz=10`), begins at hot rated-power equilibrium, and rotates all twelve drums together. The endpoint is calibrated from the 3-D worth curve to insert +0.25 dollar: a substantial but sub-prompt-critical manoeuvre. Rotation begins at 5 s, lasts 15 s, and the simulation continues to 60 s so delayed neutrons and the 268 s fuel thermal time scale both affect the response.

The 0.2 s neutron step gives 75 samples across the smooth rotation; the 1 s thermal step is an accuracy choice, not a stability limit. Thirty-one cached drum frames limit geometry rebuilds to one every 0.5 s during motion. Worth calibration itself takes several 3-D eigenvalue solves; after the first run, paste the reported endpoint into `CACHED_DRUM_TO` to skip it. This case may take a long time with the present fully spatial transient—that is intentional, because it is the acceptance benchmark for quasi-static acceleration. It is gated so **Run all** remains practical.


In [ ]:
RUN_3D_MINUTE = False
CACHED_DRUM_TO = None       # paste the calibrated endpoint here on later runs
CACHED_WORTH_PCM = None     # and its reported worth here

if RUN_3D_MINUTE:
    REFINE, NZ = 4, 10
    T_END, DT, DT_THERMAL = 60.0, 0.20, 1.0
    DRUM_FROM, DOLLARS = 90.0, 0.25
    T_START, T_RAMP, N_ANGLES = 5.0, 15.0, 31
    mats3d = hpmr_endfb8_builtin(three_d=True)
    beta = float(HPMR_KINETICS.beta.sum())

    setup0 = time.perf_counter()
    if CACHED_DRUM_TO is None:
        drum_to, achieved_pcm = hpmr_angle_for_dollars(
            DRUM_FROM, DOLLARS, refine=REFINE, nz=NZ, materials=mats3d,
            device="gpu", with_worth=True)
        print(f"Cache for later: CACHED_DRUM_TO={drum_to:.8g}; "
              f"CACHED_WORTH_PCM={achieved_pcm:.8g}")
    else:
        drum_to = float(CACHED_DRUM_TO)
        achieved_pcm = (float(CACHED_WORTH_PCM) if CACHED_WORTH_PCM is not None
                        else DOLLARS * beta * 1e5)

    problem3d = build_hpmr3d(
        refine=REFINE, nz=NZ, drum_angle_deg=DRUM_FROM,
        absorber="polar", materials=mats3d)
    ctx3d = build_hpmr_coupling(problem3d, device="gpu")
    control3d = hpmr_drum_ramp(
        problem3d, angle_from=DRUM_FROM, angle_to=drum_to,
        t_start=T_START, t_ramp=T_RAMP, n_angles=N_ANGLES,
        refine=REFINE, nz=NZ, materials=mats3d)
    setup_wall = time.perf_counter() - setup0

    cells3d = int(np.count_nonzero(problem3d.active))
    steps3d = int(round(T_END / DT))
    tau = ctx3d.thermal_materials[1].heat_capacity / sink_coefficient()
    print(f"3-D HP-MR: {cells3d:,} active cells, {11*cells3d:,} flux unknowns")
    print(f"all drums {DRUM_FROM:.2f}° -> {drum_to:.4f}°; "
          f"{achieved_pcm:+.1f} pcm = {achieved_pcm/(beta*1e5):+.3f} $")
    print(f"{T_END:g} s / {DT:g} s = {steps3d:,} neutron steps; "
          f"thermal dt={DT_THERMAL:g} s; {N_ANGLES} cached drum frames")
    print(f"rated power={RATED_POWER_W/1e6:g} MWt; thermal time constant={tau:.0f} s")
    print(f"worth calibration + geometry/frame setup: {setup_wall:.1f} s")

    t0 = time.perf_counter()
    minute3d = coupled_transient(
        ctx3d, t_end=T_END, dt=DT, dt_thermal=DT_THERMAL,
        problem_at=control3d, profile=True)
    minute3d_wall = time.perf_counter() - t0
    print_profile(minute3d, minute3d_wall)
    print(f"peak P/P0={minute3d.power.max():.6f} at "
          f"t={minute3d.times[int(np.argmax(minute3d.power))]:.1f} s")
    print(f"final P/P0={minute3d.power[-1]:.6f}; "
          f"mean fuel dT={minute3d.mean_temperature[-1]-minute3d.mean_temperature[0]:+.3f} K")
    print("\n5-second history:")
    for target in np.arange(0.0, T_END + 0.1, 5.0):
        j = int(np.argmin(np.abs(minute3d.times - target)))
        print(f"  t={minute3d.times[j]:5.1f} s  P/P0={minute3d.power[j]:9.6f}  "
              f"Tmean={minute3d.mean_temperature[j]:8.3f} K  "
              f"Tpeak={minute3d.peak_temperature[j]:8.3f} K")

    assert minute3d.steps == steps3d
    assert minute3d.counters["thermal_steps"] == int(T_END / DT_THERMAL)
    assert minute3d.counters["neutron_inner_iterations"] > 0
    assert np.all(np.isfinite(minute3d.power))
    assert np.all(np.isfinite(minute3d.temperature))
else:
    print("Skipped. Set RUN_3D_MINUTE=True for the full one-minute 3-D run.")


## Reading the report

- `startup outside transient profile` contains the coupled equilibrium and small setup overhead. `result.steady.seconds` identifies the equilibrium solve itself.
- A high `initial_eigen_solve` share supports reusing the converged coupled state instead of solving the time-zero critical state twice. Its outer/inner work is reported by the `initial_eigen_*` counters.
- `neutronics_solve` now means time-march neutron work only. A high share there means the next useful work is algorithmic acceleration—quasi-static shape updates or spatial CMFD—not more coupling plumbing.
- A high `operator_rebuild` share supports persistent/in-place feedback coefficient updates.
- A high `power_edit` share means the fused group contraction or normalization still needs work.
- A high `thermal_solve` share motivates thermal multigrid or a different thermal mesh.
- `telemetry_transfers` should equal thermal windows, **not neutron steps**. If they are equal because `dt_thermal == dt`, the cadence is a modeling choice rather than an implementation regression.
- Compare wall time only after checking neutron fixed-point sweeps and inner iterations. A faster run that performed less numerical work is not a hardware speedup.
- Do not use the small 2D A/B ratio to predict production performance. The optional one-minute 3-D leg is the acceptance workload for production scaling.
